# BERT (Colab) — Focused Best-Config Run

Single training run with the configuration determined optimal from the Exp 04 hyperparameter sweep and per-label analysis.

**Why this config:**
- `lr=1.5e-5`: monotonically best for threat F1 across all sweep runs; also best avg macro F1
- `weight_decay=0.015`: marginally but consistently better than 0.010
- `warmup_ratio=0.06`: best macro F1 in sweep summary and per-label rare classes
- `max_length=192`: BERT has genuine ROC-AUC advantage on threat (+0.003 vs DistilBERT); longer context should extend that lead on this context-dependent label
- `batch_size=16` + `grad_accum=2` → effective 32: same effective batch as DistilBERT final sweep; smaller per-step batch needed due to BERT's 109M params vs 67M
- `BCEWithLogitsLoss` no pos_weight: confirmed best in Exp D+E across all 6 loss variants
- Threshold grid extended to 0.995 with 0.005 step: all rare-label optima cluster at 0.94–0.95; finer search may find better calibration

**Fixed config:**
- model: `bert-base-uncased`, `problem_type='multi_label_classification'`
- data split: `validation_fraction=0.1`, `random_state=42`, `use_iterative_stratify=True`, `rebalance_train=False`
- tokenization: `max_length=192`, `padding='max_length'`, `truncation=True`
- training: `batch_size=16`, `gradient_accumulation_steps=2` (effective 32), `max_epochs=5`
- optimization: `AdamW`, `learning_rate=1.5e-5`, `weight_decay=0.015`, `linear_with_warmup`, `warmup_ratio=0.06`
- regularization: early stopping on validation loss (`patience=2`, `min_delta=0.0`, restore best weights)
- loss: `BCEWithLogitsLoss` with `pos_weight=None`
- precision/perf: bf16 mixed precision if supported
- evaluation: per-label tuned thresholds on `np.arange(0.05, 0.996, 0.005)` (finer grid vs prior runs)
- reproducibility: torch/numpy seed = 42

In [ ]:
# Colab setup: install dependencies
!pip -q install torch transformers pandas numpy matplotlib scikit-learn seaborn iterative-stratification

In [ ]:
# Mount Google Drive and set paths
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ===== CHANGE THIS to your repo folder in Drive =====
PROJECT_ROOT = Path('/content/drive/MyDrive/cmpe258/jigsaw-toxic-comment-classification-challenge')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'PROJECT_ROOT not found: {PROJECT_ROOT}. Upload/clone your repo to this path first.'
    )

NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
ARTIFACT_DIR = NOTEBOOKS_DIR / 'bert' / 'output' / 'bert_exp_05_focused'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(NOTEBOOKS_DIR))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('ARTIFACT_DIR:', ARTIFACT_DIR)

In [ ]:
import contextlib
import math
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import confusion_matrix, f1_score, roc_curve, auc

from preprocessing.text_preprocessing import LABEL_COLUMNS, preprocess_for_bert
from metrics_helpers import multilabel_evaluation_report, torch_parameter_count

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

RARE_LABELS = ['severe_toxic', 'threat', 'identity_hate']
COMMON_LABELS = ['toxic', 'obscene', 'insult']
LABEL_COLORS = {
    'toxic': '#e74c3c',
    'severe_toxic': '#c0392b',
    'obscene': '#e67e22',
    'threat': '#8e44ad',
    'insult': '#2980b9',
    'identity_hate': '#16a085',
}


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def binary_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    denom = 2 * tp + fp + fn
    return float((2 * tp) / denom) if denom > 0 else 0.0


def tune_per_label_thresholds(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    labels,
    threshold_grid: np.ndarray,
):
    best_thresholds = {}
    rows = []
    for j, label in enumerate(labels):
        y_true_j = y_true[:, j]
        y_prob_j = y_prob[:, j]
        best_t, best_f1 = 0.5, -1.0
        for t in threshold_grid:
            y_pred_j = (y_prob_j >= t).astype(np.int64)
            f1_j = binary_f1(y_true_j, y_pred_j)
            if f1_j > best_f1:
                best_f1 = f1_j
                best_t = float(t)
        best_thresholds[label] = best_t
        rows.append({'label': label, 'best_threshold': round(best_t, 4), 'best_f1_on_val': round(best_f1, 6)})
    return best_thresholds, pd.DataFrame(rows)


def apply_thresholds(y_prob: np.ndarray, labels, thresholds: dict) -> np.ndarray:
    out = np.zeros_like(y_prob, dtype=np.int64)
    for j, label in enumerate(labels):
        out[:, j] = (y_prob[:, j] >= thresholds[label]).astype(np.int64)
    return out


def make_confusion_artifacts(y_true: np.ndarray, y_pred: np.ndarray, labels):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    per_label_rows = []
    for j, label in enumerate(labels):
        cm = confusion_matrix(y_true[:, j], y_pred[:, j], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        per_label_rows.append({'label': label, 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)})
    per_label_df = pd.DataFrame(per_label_rows)
    agg_cm = confusion_matrix(y_true.ravel(), y_pred.ravel(), labels=[0, 1])
    agg_tn, agg_fp, agg_fn, agg_tp = agg_cm.ravel()
    aggregate_df = pd.DataFrame([{'tn': int(agg_tn), 'fp': int(agg_fp), 'fn': int(agg_fn), 'tp': int(agg_tp)}])
    return per_label_df, aggregate_df


class EncDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item


def collate(batch):
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0] if k != 'labels'}
    out['labels'] = torch.stack([b['labels'] for b in batch], dim=0)
    return out


def enc_dict(enc):
    # BERT includes token_type_ids; include all valid keys
    keys = [k for k in ('input_ids', 'attention_mask', 'token_type_ids') if k in enc]
    return {k: enc[k] for k in keys}

In [ ]:
# Configuration — single focused run
DEVICE = pick_device()
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

torch.manual_seed(42)
np.random.seed(42)

# model
MODEL_NAME = 'bert-base-uncased'
PROBLEM_TYPE = 'multi_label_classification'

# data
VALIDATION_FRACTION = 0.1
RANDOM_STATE = 42
USE_ITERATIVE_STRATIFY = True
REBALANCE_TRAIN = False

# tokenization — extended to 192 to give BERT more context for threat/identity_hate
MAX_LENGTH = 192

# training
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2  # effective batch = 32
MAX_EPOCHS = 5
LR = 1.5e-5          # best for threat F1 and macro F1 in sweep
WEIGHT_DECAY = 0.015  # marginally best in sweep
WARMUP_RATIO = 0.06   # best macro F1 in sweep
MAX_GRAD_NORM = 1.0

# regularization
EARLY_STOP_ENABLED = True
EARLY_STOP_PATIENCE = 2
EARLY_STOP_MIN_DELTA = 0.0

# loss
USE_POS_WEIGHT = False

# precision/perf
USE_BF16 = True
USE_TORCH_COMPILE = False
NUM_WORKERS = 2

# evaluation — finer threshold grid extended to 0.995
THRESHOLD_GRID = np.arange(0.05, 0.996, 0.005)

# reproducibility
TORCH_SEED = 42
NUMPY_SEED = 42

_bf16_ok = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
USE_AMP = bool(USE_BF16 and _bf16_ok)
if USE_BF16 and DEVICE.type == 'cuda' and not _bf16_ok:
    print('USE_BF16 requested but bf16 not supported on this GPU; training in full precision.')

print(f'Device:         {DEVICE}')
print(f'AMP (bf16):     {USE_AMP}')
print(f'Model:          {MODEL_NAME}')
print(f'Max length:     {MAX_LENGTH}')
print(f'LR:             {LR}')
print(f'Weight decay:   {WEIGHT_DECAY}')
print(f'Warmup ratio:   {WARMUP_RATIO}')
print(f'Effective batch:{BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}')
print(f'Threshold steps:{len(THRESHOLD_GRID)} ({THRESHOLD_GRID[0]:.3f}–{THRESHOLD_GRID[-1]:.3f}, step={THRESHOLD_GRID[1]-THRESHOLD_GRID[0]:.3f})')

In [ ]:
# Training — single full-data run
torch.manual_seed(TORCH_SEED)
np.random.seed(NUMPY_SEED)
pin = DEVICE.type == 'cuda'

print('Loading and tokenizing data...')
run_data = preprocess_for_bert(
    validation_fraction=VALIDATION_FRACTION,
    random_state=RANDOM_STATE,
    max_length=MAX_LENGTH,
    return_tensors='pt',
    max_train_samples=None,
    max_val_samples=None,
    use_iterative_stratify=USE_ITERATIVE_STRATIFY,
    rebalance_train=REBALANCE_TRAIN,
    print_diagnostics=True,
)

train_enc = enc_dict(run_data.train_encodings)
val_enc = enc_dict(run_data.val_encodings)
y_train = np.asarray(run_data.y_train, dtype=np.float32)
y_val = np.asarray(run_data.y_val, dtype=np.int64)

print(f'Train rows: {y_train.shape[0]} | Val rows: {y_val.shape[0]}')

train_loader = DataLoader(
    EncDataset(train_enc, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate,
    num_workers=NUM_WORKERS,
    pin_memory=pin,
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    EncDataset(val_enc, y_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate,
    num_workers=NUM_WORKERS,
    pin_memory=pin,
    persistent_workers=NUM_WORKERS > 0,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_COLUMNS),
    problem_type=PROBLEM_TYPE,
).to(DEVICE)

num_params = torch_parameter_count(model)
print(f'Parameters: {num_params:,}')

no_decay = ['bias', 'LayerNorm.weight']
optimizer = torch.optim.AdamW(
    [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ],
    lr=LR,
)

steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
num_training_steps = steps_per_epoch * MAX_EPOCHS
warmup_steps = max(0, min(int(WARMUP_RATIO * num_training_steps), num_training_steps - 1))
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=num_training_steps
)

autocast_ctx = (
    torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=True)
    if USE_AMP else contextlib.nullcontext()
)

# Training state
best_val_loss = float('inf')
best_state_cpu = None
best_epoch = 0
patience_left = EARLY_STOP_PATIENCE
epochs_ran = 0
early_stopped = False
train_time_s = 0.0
inference_time_s = 0.0

# Per-epoch loss history for plotting
epoch_train_losses = []
epoch_val_losses = []

print(f'\nTraining: {num_training_steps} total steps, {warmup_steps} warmup steps')
print('-' * 65)

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss_sum = 0.0
    epoch_batches = 0
    t0 = time.perf_counter()
    optimizer.zero_grad(set_to_none=True)
    micro = 0

    for batch in train_loader:
        batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
        labels = batch.pop('labels')
        with autocast_ctx:
            logits = model(**batch).logits
            loss = F.binary_cross_entropy_with_logits(logits, labels) / GRADIENT_ACCUMULATION_STEPS
        loss.backward()
        micro += 1
        epoch_loss_sum += float(loss.item()) * GRADIENT_ACCUMULATION_STEPS
        epoch_batches += 1

        if micro % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    if micro % GRADIENT_ACCUMULATION_STEPS != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    train_time_s += time.perf_counter() - t0
    train_loss = epoch_loss_sum / max(epoch_batches, 1)

    model.eval()
    val_loss_sum = 0.0
    val_batches = 0
    t1 = time.perf_counter()
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
            labels = batch.pop('labels')
            with autocast_ctx:
                logits = model(**batch).logits
                vloss = F.binary_cross_entropy_with_logits(logits, labels)
            val_loss_sum += float(vloss.item())
            val_batches += 1
    inference_time_s += time.perf_counter() - t1

    val_loss = val_loss_sum / max(val_batches, 1)
    epochs_ran = epoch
    epoch_train_losses.append(train_loss)
    epoch_val_losses.append(val_loss)

    improved = (best_val_loss - val_loss) > EARLY_STOP_MIN_DELTA
    if improved:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_left = EARLY_STOP_PATIENCE
        best_state_cpu = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        if EARLY_STOP_ENABLED:
            patience_left -= 1
            if patience_left <= 0:
                early_stopped = True
                print(f'  early stop triggered at epoch {epoch}')
                break

    star = ' *' if improved else ''
    print(
        f'  epoch {epoch:02d} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f} | '
        f'best_val={best_val_loss:.5f} | patience={patience_left}{star}'
    )

if best_state_cpu is not None:
    model.load_state_dict(best_state_cpu)
    print(f'\nRestored best weights from epoch {best_epoch}')

print(f'\nTraining complete: {epochs_ran} epochs, {train_time_s:.1f}s train, early_stopped={early_stopped}')

In [ ]:
# Inference, threshold tuning, and save artifacts
model.eval()
probs_all = []
t_inf = time.perf_counter()
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
        _ = batch.pop('labels')
        with autocast_ctx:
            logits = model(**batch).logits
        probs_all.append(torch.sigmoid(logits).float().detach().cpu().numpy())
inference_time_s += time.perf_counter() - t_inf

y_prob = np.vstack(probs_all)

# Baseline (threshold=0.5)
y_pred_base = (y_prob >= 0.5).astype(np.int64)
per_label_base, base_summary = multilabel_evaluation_report(y_val, y_pred_base, y_prob, LABEL_COLUMNS)

# Tuned thresholds (finer grid, extended range)
best_thresholds, thresholds_df = tune_per_label_thresholds(y_val, y_prob, LABEL_COLUMNS, THRESHOLD_GRID)
y_pred_tuned = apply_thresholds(y_prob, LABEL_COLUMNS, best_thresholds)
per_label_tuned, tuned_summary = multilabel_evaluation_report(y_val, y_pred_tuned, y_prob, LABEL_COLUMNS)

# Confusion matrices
conf_per_label_base_df, conf_agg_base_df = make_confusion_artifacts(y_val, y_pred_base, LABEL_COLUMNS)
conf_per_label_tuned_df, conf_agg_tuned_df = make_confusion_artifacts(y_val, y_pred_tuned, LABEL_COLUMNS)

# Summary row
summary = {
    'model_name': MODEL_NAME,
    'problem_type': PROBLEM_TYPE,
    'validation_fraction': VALIDATION_FRACTION,
    'random_state': RANDOM_STATE,
    'use_iterative_stratify': USE_ITERATIVE_STRATIFY,
    'rebalance_train': REBALANCE_TRAIN,
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    'effective_batch_size': BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    'max_epochs': MAX_EPOCHS,
    'epochs_ran': epochs_ran,
    'best_epoch': best_epoch,
    'early_stopped': early_stopped,
    'learning_rate': LR,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
    'warmup_steps': warmup_steps,
    'max_grad_norm': MAX_GRAD_NORM,
    'early_stop_patience': EARLY_STOP_PATIENCE,
    'early_stop_min_delta': EARLY_STOP_MIN_DELTA,
    'loss_type': 'BCEWithLogitsLoss',
    'use_pos_weight': USE_POS_WEIGHT,
    'use_amp': USE_AMP,
    'num_parameters': num_params,
    'actual_train_samples': int(y_train.shape[0]),
    'actual_val_samples': int(y_val.shape[0]),
    'train_time_s': round(train_time_s, 2),
    'inference_time_s': round(inference_time_s, 2),
    'best_val_loss': best_val_loss,
    'train_loss_last': epoch_train_losses[-1],
    'val_loss_last': epoch_val_losses[-1],
    'baseline_micro_f1': base_summary['f1_micro'],
    'baseline_macro_f1': base_summary['f1_macro'],
    'baseline_samples_f1': float(f1_score(y_val, y_pred_base, average='samples', zero_division=0)),
    'tuned_micro_f1': tuned_summary['f1_micro'],
    'tuned_macro_f1': tuned_summary['f1_macro'],
    'tuned_samples_f1': float(f1_score(y_val, y_pred_tuned, average='samples', zero_division=0)),
    'threshold_grid_start': float(THRESHOLD_GRID[0]),
    'threshold_grid_end': float(THRESHOLD_GRID[-1]),
    'threshold_grid_step': round(float(THRESHOLD_GRID[1] - THRESHOLD_GRID[0]), 4),
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(ARTIFACT_DIR / 'bert_exp_05_summary.csv', index=False)
per_label_tuned.to_csv(ARTIFACT_DIR / 'bert_exp_05_per_label.csv', index=False)
per_label_base.to_csv(ARTIFACT_DIR / 'bert_exp_05_per_label_baseline.csv', index=False)
thresholds_df.to_csv(ARTIFACT_DIR / 'bert_exp_05_thresholds.csv', index=False)
conf_per_label_base_df.to_csv(ARTIFACT_DIR / 'bert_exp_05_confusion_per_label_baseline.csv', index=False)
conf_per_label_tuned_df.to_csv(ARTIFACT_DIR / 'bert_exp_05_confusion_per_label_tuned.csv', index=False)
conf_agg_base_df.to_csv(ARTIFACT_DIR / 'bert_exp_05_confusion_aggregate_baseline.csv', index=False)
conf_agg_tuned_df.to_csv(ARTIFACT_DIR / 'bert_exp_05_confusion_aggregate_tuned.csv', index=False)

print(f'baseline  micro={base_summary["f1_micro"]:.4f}  macro={base_summary["f1_macro"]:.4f}')
print(f'tuned     micro={tuned_summary["f1_micro"]:.4f}  macro={tuned_summary["f1_macro"]:.4f}')
print('Artifacts saved to:', ARTIFACT_DIR)
thresholds_df

In [ ]:
# Graph 1: Training and validation loss curves
epochs_x = list(range(1, epochs_ran + 1))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(epochs_x, epoch_train_losses, marker='o', color='#2980b9', label='Train loss')
ax.plot(epochs_x, epoch_val_losses, marker='s', color='#e74c3c', label='Val loss')
ax.axvline(best_epoch, color='gray', linestyle='--', linewidth=1, label=f'Best epoch ({best_epoch})')
ax.set_title(f'BERT Training & Validation Loss — {MODEL_NAME} (max_len={MAX_LENGTH}, lr={LR})')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE Loss')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'graph_01_loss_curves.png', dpi=150)
plt.show()

In [ ]:
# Graph 2: Per-label F1 — baseline vs tuned, with tuning gain annotation
labels_order = list(LABEL_COLUMNS)
base_f1 = per_label_base.set_index('label').loc[labels_order, 'f1'].values
tuned_f1 = per_label_tuned.set_index('label').loc[labels_order, 'f1'].values
gain = tuned_f1 - base_f1

x = np.arange(len(labels_order))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars_base = ax.bar(x - w/2, base_f1, width=w, label='Baseline (t=0.5)', color='#aed6f1', edgecolor='white')
bars_tuned = ax.bar(x + w/2, tuned_f1, width=w, label='Tuned threshold', color='#2980b9', edgecolor='white')

for i, (b, t, g) in enumerate(zip(base_f1, tuned_f1, gain)):
    ax.text(x[i] - w/2, b + 0.008, f'{b:.3f}', ha='center', va='bottom', fontsize=8, color='#555')
    ax.text(x[i] + w/2, t + 0.008, f'{t:.3f}', ha='center', va='bottom', fontsize=8, color='#1a5276')
    ax.text(x[i] + w/2, t + 0.035, f'+{g:.3f}', ha='center', va='bottom', fontsize=7.5, color='#117a65')

ax.set_xticks(x)
ax.set_xticklabels([l.replace('_', '\n') for l in labels_order])
ax.set_ylim(0, 1.0)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Label F1: Baseline vs Tuned Threshold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Shade rare label region
rare_idx = [labels_order.index(l) for l in RARE_LABELS]
for ri in rare_idx:
    ax.axvspan(ri - 0.5, ri + 0.5, alpha=0.05, color='purple')
ax.text(rare_idx[0] - 0.4, 0.97, 'rare labels', fontsize=8, color='purple', alpha=0.6)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'graph_02_per_label_f1.png', dpi=150)
plt.show()

In [ ]:
# Graph 3: Per-label ROC-AUC
roc_auc = per_label_tuned.set_index('label').loc[labels_order, 'roc_auc'].values
colors = [LABEL_COLORS[l] for l in labels_order]

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(labels_order, roc_auc, color=colors, edgecolor='white', width=0.55)
for bar, val in zip(bars, roc_auc):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 0.005,
            f'{val:.4f}', ha='center', va='top', fontsize=9, color='white', fontweight='bold')

ax.set_ylim(0.96, 1.0)
ax.set_ylabel('ROC-AUC')
ax.set_title('Per-Label ROC-AUC (tuned threshold predictions)')
ax.set_xticklabels([l.replace('_', '\n') for l in labels_order])
ax.axhline(np.mean(roc_auc), color='gray', linestyle='--', linewidth=1, label=f'Mean = {np.mean(roc_auc):.4f}')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'graph_03_per_label_roc_auc.png', dpi=150)
plt.show()

In [ ]:
# Graph 4: Precision vs Recall per label (tuned) — scatter with F1 iso-curves
prec = per_label_tuned.set_index('label').loc[labels_order, 'precision'].values
rec  = per_label_tuned.set_index('label').loc[labels_order, 'recall'].values
f1v  = per_label_tuned.set_index('label').loc[labels_order, 'f1'].values

fig, ax = plt.subplots(figsize=(8, 6))

# F1 iso-curves
r_grid = np.linspace(0.01, 1.0, 300)
for f1_iso in [0.4, 0.5, 0.6, 0.7, 0.8]:
    p_iso = f1_iso * r_grid / (2 * r_grid - f1_iso + 1e-9)
    mask = (p_iso >= 0) & (p_iso <= 1)
    ax.plot(r_grid[mask], p_iso[mask], color='lightgray', linewidth=0.8, linestyle='--')
    idx = np.argmin(np.abs(r_grid - 0.82))
    if mask[idx]:
        ax.text(r_grid[idx], p_iso[idx], f'F1={f1_iso}', fontsize=7, color='gray')

for i, label in enumerate(labels_order):
    c = LABEL_COLORS[label]
    ax.scatter(rec[i], prec[i], s=180, color=c, zorder=5, edgecolors='white', linewidth=1)
    offset = (0.012, 0.012)
    ax.annotate(
        f"{label.replace('_', chr(10))}\nF1={f1v[i]:.3f}",
        (rec[i], prec[i]),
        xytext=(rec[i] + offset[0], prec[i] + offset[1]),
        fontsize=8, color=c,
        arrowprops=dict(arrowstyle='-', color=c, lw=0.8),
    )

ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision vs Recall per Label (tuned thresholds)')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'graph_04_precision_recall.png', dpi=150)
plt.show()

In [ ]:
# Graph 5: Per-label confusion matrix heatmaps (tuned)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for i, label in enumerate(labels_order):
    row = conf_per_label_tuned_df[conf_per_label_tuned_df['label'] == label].iloc[0]
    cm_arr = np.array([[row['tn'], row['fp']], [row['fn'], row['tp']]], dtype=np.int64)
    # Normalise by row (true label counts)
    cm_norm = cm_arr.astype(float) / cm_arr.sum(axis=1, keepdims=True).clip(min=1)

    sns.heatmap(
        cm_norm, annot=cm_arr, fmt='d', cmap='Blues',
        xticklabels=['Pred 0', 'Pred 1'],
        yticklabels=['True 0', 'True 1'],
        ax=axes[i], vmin=0, vmax=1,
        cbar=False,
    )
    t_opt = best_thresholds.get(label, 0.5)
    f1_l = per_label_tuned.set_index('label').loc[label, 'f1']
    axes[i].set_title(
        f'{label}\nt={t_opt:.3f} | F1={f1_l:.3f}',
        fontsize=10, color=LABEL_COLORS[label]
    )

fig.suptitle('Per-Label Confusion Matrices — Tuned Thresholds (row-normalised, counts shown)', fontsize=12)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'graph_05_confusion_matrices_tuned.png', dpi=150)
plt.show()

In [ ]:
# Graph 6: Threshold sensitivity curves for all labels
# Shows F1 vs threshold across the full grid — highlights where the optimum sits
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False)
axes = axes.ravel()

for i, label in enumerate(labels_order):
    j = list(LABEL_COLUMNS).index(label)
    f1_curve = [
        binary_f1(y_val[:, j], (y_prob[:, j] >= t).astype(np.int64))
        for t in THRESHOLD_GRID
    ]
    opt_t = best_thresholds[label]
    opt_f1 = max(f1_curve)

    axes[i].plot(THRESHOLD_GRID, f1_curve, color=LABEL_COLORS[label], linewidth=1.5)
    axes[i].axvline(opt_t, color='gray', linestyle='--', linewidth=1)
    axes[i].axvline(0.5, color='lightgray', linestyle=':', linewidth=1)
    axes[i].scatter([opt_t], [opt_f1], color=LABEL_COLORS[label], s=60, zorder=5)
    axes[i].set_title(f'{label}\nopt t={opt_t:.3f}, F1={opt_f1:.3f}', fontsize=9)
    axes[i].set_xlabel('Threshold')
    axes[i].set_ylabel('F1')
    axes[i].set_xlim(0.05, 1.0)
    axes[i].set_ylim(0, 1.0)
    axes[i].grid(alpha=0.3)

    # Annotate default threshold performance
    default_idx = np.argmin(np.abs(THRESHOLD_GRID - 0.5))
    axes[i].text(0.51, f1_curve[default_idx] + 0.03, f'@0.5={f1_curve[default_idx]:.3f}',
                 fontsize=7.5, color='gray')

fig.suptitle('F1 vs Decision Threshold per Label (finer 0.005-step grid)', fontsize=12)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'graph_06_threshold_sensitivity.png', dpi=150)
plt.show()

In [ ]:
# Graph 7: ROC curves per label
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, label in enumerate(labels_order):
    j = list(LABEL_COLUMNS).index(label)
    fpr, tpr, _ = roc_curve(y_val[:, j], y_prob[:, j])
    roc_auc_val = auc(fpr, tpr)
    opt_t = best_thresholds[label]

    # Mark the operating point at optimal threshold
    y_pred_opt = (y_prob[:, j] >= opt_t).astype(np.int64)
    from sklearn.metrics import recall_score
    from sklearn.metrics import precision_score as ps
    tp_rate = recall_score(y_val[:, j], y_pred_opt, zero_division=0)
    fp_rate = (((y_val[:, j] == 0) & (y_pred_opt == 1)).sum() /
               max((y_val[:, j] == 0).sum(), 1))

    axes[i].plot(fpr, tpr, color=LABEL_COLORS[label], linewidth=1.8, label=f'AUC={roc_auc_val:.4f}')
    axes[i].plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5)
    axes[i].scatter([fp_rate], [tp_rate], s=80, color=LABEL_COLORS[label],
                    zorder=5, edgecolors='white', linewidth=1,
                    label=f'op. point (t={opt_t:.3f})')
    axes[i].set_title(label, fontsize=10, color=LABEL_COLORS[label])
    axes[i].set_xlabel('FPR')
    axes[i].set_ylabel('TPR')
    axes[i].legend(fontsize=8)
    axes[i].grid(alpha=0.3)

fig.suptitle('ROC Curves per Label (dot = operating point at tuned threshold)', fontsize=12)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'graph_07_roc_curves.png', dpi=150)
plt.show()

In [ ]:
# Graph 8: BERT vs DistilBERT — per-label F1 comparison
# DistilBERT reference values from the final training-size sweep at full data (143k)
distilbert_f1 = {
    'toxic':         0.8408,
    'severe_toxic':  0.5431,
    'obscene':       0.8445,
    'threat':        0.5679,
    'insult':        0.7718,
    'identity_hate': 0.6082,
}
distilbert_roc = {
    'toxic':         0.9869,
    'severe_toxic':  0.9917,
    'obscene':       0.9923,
    'threat':        0.9861,
    'insult':        0.9890,
    'identity_hate': 0.9880,
}

bert_f1 = per_label_tuned.set_index('label').loc[labels_order, 'f1'].to_dict()
bert_roc = per_label_tuned.set_index('label').loc[labels_order, 'roc_auc'].to_dict()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 comparison
x = np.arange(len(labels_order))
w = 0.35
db_f1_vals = [distilbert_f1[l] for l in labels_order]
b_f1_vals  = [bert_f1[l] for l in labels_order]
delta_f1   = [b - d for b, d in zip(b_f1_vals, db_f1_vals)]

axes[0].bar(x - w/2, db_f1_vals, width=w, label='DistilBERT (final sweep)', color='#aed6f1', edgecolor='white')
axes[0].bar(x + w/2, b_f1_vals,  width=w, label='BERT (this run)',          color='#1a5276', edgecolor='white')
for i, (d, b, g) in enumerate(zip(db_f1_vals, b_f1_vals, delta_f1)):
    col = '#117a65' if g >= 0 else '#922b21'
    sign = '+' if g >= 0 else ''
    axes[0].text(x[i] + w/2, b + 0.01, f'{sign}{g:.3f}', ha='center', va='bottom', fontsize=8, color=col)
axes[0].set_xticks(x)
axes[0].set_xticklabels([l.replace('_', '\n') for l in labels_order])
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel('F1 (tuned)')
axes[0].set_title('Per-Label F1: BERT vs DistilBERT')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

# ROC-AUC comparison
db_roc_vals = [distilbert_roc[l] for l in labels_order]
b_roc_vals  = [bert_roc[l] for l in labels_order]
delta_roc   = [b - d for b, d in zip(b_roc_vals, db_roc_vals)]

axes[1].bar(x - w/2, db_roc_vals, width=w, label='DistilBERT', color='#aed6f1', edgecolor='white')
axes[1].bar(x + w/2, b_roc_vals,  width=w, label='BERT',       color='#1a5276', edgecolor='white')
for i, (d, b, g) in enumerate(zip(db_roc_vals, b_roc_vals, delta_roc)):
    col = '#117a65' if g >= 0 else '#922b21'
    sign = '+' if g >= 0 else ''
    axes[1].text(x[i] + w/2, b + 0.0002, f'{sign}{g:.4f}', ha='center', va='bottom', fontsize=7.5, color=col)
axes[1].set_xticks(x)
axes[1].set_xticklabels([l.replace('_', '\n') for l in labels_order])
axes[1].set_ylim(0.97, 1.0)
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Per-Label ROC-AUC: BERT vs DistilBERT')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('BERT (max_len=192) vs DistilBERT (max_len=128) — Full Dataset', fontsize=12)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'graph_08_bert_vs_distilbert.png', dpi=150)
plt.show()

# Print delta table
delta_df = pd.DataFrame({
    'label': labels_order,
    'distilbert_f1': db_f1_vals,
    'bert_f1': [round(v, 4) for v in b_f1_vals],
    'delta_f1': [round(g, 4) for g in delta_f1],
    'distilbert_roc': db_roc_vals,
    'bert_roc': [round(v, 4) for v in b_roc_vals],
    'delta_roc': [round(g, 4) for g in delta_roc],
})
delta_df.to_csv(ARTIFACT_DIR / 'bert_exp_05_vs_distilbert.csv', index=False)
delta_df

In [ ]:
# Summary tables
print('=== Run Summary ===')
key_cols = [
    'model_name', 'max_length', 'learning_rate', 'weight_decay', 'warmup_ratio',
    'effective_batch_size', 'epochs_ran', 'best_epoch', 'early_stopped',
    'train_time_s', 'inference_time_s', 'num_parameters',
    'baseline_micro_f1', 'baseline_macro_f1',
    'tuned_micro_f1', 'tuned_macro_f1',
]
display(summary_df[key_cols].T.rename(columns={0: 'value'}))

print('\n=== Per-Label Tuned Metrics ===')
display(per_label_tuned.round(4))

print('\n=== Optimal Thresholds ===')
display(thresholds_df)